In [ ]:
import os
import zipfile

if not os.path.exists("config.json") and os.path.exists("/content/predict_tc_main/config.json"):
    os.chdir("/content/predict_tc_main")

if not os.path.exists("config.json") and os.path.exists("/content"):
    from google.colab import files
    uploaded = files.upload()
    zip_name = None
    for name in uploaded:
        if name.endswith(".zip"):
            zip_name = name
    assert zip_name, "need file predict_tc_main.zip"
    with zipfile.ZipFile(zip_name) as z:
        if "config.json" in z.namelist():
            target = "/content/predict_tc_main"
        else:
            target = "/content"
        z.extractall(target)
    os.chdir("/content/predict_tc_main")

print(os.getcwd())

In [ ]:
%pip install -q alignn==2026.8.11 jarvis-tools mace-torch

In [ ]:
import copy
import glob
import gzip
import json
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt

from ase import Atoms as AseAtoms
from jarvis.core.atoms import Atoms
from alignn.torch_graph_builder import build_pure_torch_graph, batch_torch_graph_pairs
from alignn.models.alignn_atomwise_pure import ALIGNNAtomWisePure, ALIGNNAtomWisePureConfig, scatter_mean

cfg = json.load(open("config.json"))

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

device

## Data

In [ ]:
data = json.load(gzip.open("data/tc_data.json.gz", "rt"))
groups = json.load(gzip.open("data/groups.json.gz", "rt"))

by_id = {}
for record in data:
    by_id[record["id"]] = record

def tc_values(ids):
    values = []
    for i in ids:
        values.append(by_id[i]["Tc_K"])
    return np.array(values, float)

def to_graph(atoms):
    return build_pure_torch_graph(
        atoms,
        two_body_cutoff=cfg["cutoff"],
        max_neighbors=cfg["neighbors"],
        atom_features="cgcnn",
        compute_line_graph=True,
    )

def global_features(atoms, pressure):
    n_atoms = len(atoms.elements)
    volume_per_atom = atoms.volume / n_atoms
    hydrogen_fraction = atoms.elements.count("H") / n_atoms
    return [pressure, volume_per_atom, hydrogen_fraction]

graphs = {}
extra = {}
for record in data:
    atoms = Atoms(**record["atoms"])
    graphs[record["id"]] = to_graph(atoms)
    extra[record["id"]] = global_features(atoms, record["pressure_GPa"])

rows = []
for gi, g in enumerate(groups):
    rows.append({"група": gi, "train": len(g["train"]), "val": len(g["val"]), "test": len(g["test"])})
pd.DataFrame(rows)

In [ ]:
class TcData(torch.utils.data.Dataset):
    def __init__(self, ids, y, xm, xs):
        self.ids = ids
        self.y = y.astype(np.float32)
        self.x = ((np.array([extra[i] for i in ids], float) - xm) / xs).astype(np.float32)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        g, lg = graphs[self.ids[i]]
        return g, lg, self.x[i], self.y[i], self.ids[i]

def collate(batch):
    graph_pairs = [(b[0], b[1]) for b in batch]
    g, lg = batch_torch_graph_pairs(graph_pairs)
    x = torch.as_tensor(np.array([b[2] for b in batch]))
    y = torch.as_tensor(np.array([b[3] for b in batch]))
    ids = [b[4] for b in batch]
    return g, lg, x, y, ids

def make_loader(dataset, shuffle, seed, batch_size):
    generator = torch.Generator().manual_seed(seed)
    return torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate, generator=generator
    )

def metrics(e):
    error = e.Tc_pred - e.Tc_true
    mae = error.abs().mean()
    rmse = float(np.sqrt((error ** 2).mean()))
    ss_res = (error ** 2).sum()
    ss_tot = ((e.Tc_true - e.Tc_true.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot
    return {"MAE, K": mae, "RMSE, K": rmse, "R²": r2}

## 1. ALIGNN×3

15 networks: 5 groups × 3 configs

In [ ]:
class TcModel(nn.Module):
    def __init__(self, c):
        super().__init__()
        encoder = ALIGNNAtomWisePure(ALIGNNAtomWisePureConfig(
            name="alignn_atomwise_pure",
            alignn_layers=c["alignn_layers"],
            gcn_layers=c["gcn_layers"],
            atom_input_features=92 + 3,
            hidden_features=c["hidden"],
            embedding_features=c["embedding"],
            calculate_gradient=False,
            gradwise_weight=0.0,
            use_penalty=False,
            energy_mult_natoms=False,
        ))
        self.atom_embedding = encoder.atom_embedding
        self.edge_embedding = encoder.edge_embedding
        self.angle_embedding = encoder.angle_embedding
        self.alignn_layers = encoder.alignn_layers
        self.gcn_layers = encoder.gcn_layers

        hidden = c["hidden"]
        head_hidden = c["head_hidden"]
        dropout = c["dropout"]
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, head_hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, 1),
        )

    def forward(self, g, lg, x_extra):
        x = self.atom_embedding(torch.cat([g.ndata["atom_features"], x_extra[g.node_batch_id]], 1))
        y = self.edge_embedding(torch.norm(g.edata["r"], dim=1))
        z = self.angle_embedding(lg.edata["h"])
        for layer in self.alignn_layers:
            x, y, z = layer(g, lg, x, y, z)
        for layer in self.gcn_layers:
            x, y = layer(g, x, y)
        pooled = scatter_mean(x, g.node_batch_id, g.batch_size)
        return self.head(pooled).squeeze(-1)

def run(model, loader, opt=None, sched=None):
    model.train(opt is not None)
    predictions = []
    targets = []
    ids = []
    with torch.set_grad_enabled(opt is not None):
        for g, lg, x, y, batch_ids in loader:
            g, lg, x, y = g.to(device), lg.to(device), x.to(device), y.to(device)
            out = model(g, lg, x)
            if opt is not None:
                loss = nn.functional.huber_loss(out, y, delta=cfg["huber_delta"])
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step()
                sched.step()
            predictions.append(out.detach().cpu().numpy())
            targets.append(y.cpu().numpy())
            ids += batch_ids
    return np.concatenate(predictions), np.concatenate(targets), ids

def train_alignn(gi, seed, c):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    g = groups[gi]
    use_log = c["target"] == "log"

    def forward_transform(values):
        if use_log:
            return np.log1p(values)
        return values

    def inverse_transform(values):
        values = np.asarray(values, float) * std + mean
        if use_log:
            values = np.expm1(values)
        return np.clip(values, 0, None)

    tc = forward_transform(tc_values(g["train"]))
    mean = float(tc.mean())
    std = float(tc.std() + 1e-8)

    gx = np.array([extra[i] for i in g["train"]], float)
    xm = gx.mean(0)
    xs = gx.std(0) + 1e-8

    def build_loader(part, shuffle):
        y = (forward_transform(tc_values(g[part])) - mean) / std
        dataset = TcData(g[part], y, xm, xs)
        return make_loader(dataset, shuffle, seed, c["batch_size"])

    train_dl = build_loader("train", True)
    val_dl = build_loader("val", False)

    model = TcModel(c).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=c["lr"], weight_decay=c["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=c["lr"], epochs=c["epochs"], steps_per_epoch=len(train_dl)
    )

    best_state = copy.deepcopy(model.state_dict())
    best_score = np.inf
    best_epoch = 0

    for epoch in range(c["epochs"]):
        run(model, train_dl, opt, sched)
        p, y, _ = run(model, val_dl)
        score = float(np.abs(inverse_transform(p) - inverse_transform(y)).mean())
        if score < best_score:
            best_state = copy.deepcopy(model.state_dict())
            best_score = score
            best_epoch = epoch + 1

    model.load_state_dict(best_state)
    p, y, ids = run(model, build_loader("test", False))
    pred = pd.DataFrame({"id": ids, "Tc_true": inverse_transform(y), "Tc_pred": inverse_transform(p)})
    scale = dict(target=c["target"], mean=mean, std=std, xm=xm.tolist(), xs=xs.tolist())
    return model, scale, pred, best_epoch

In [ ]:
os.makedirs("models", exist_ok=True)

alignn_runs = []
for gi in range(len(groups)):
    for name, change in cfg["members"].items():
        member_cfg = dict(cfg)
        member_cfg.update(change)

        model, scale, pred, best_epoch = train_alignn(gi, cfg["seed"], member_cfg)
        torch.save({"state": model.state_dict(), "member": name, **scale}, f"models/g{gi}_{name}.pt")

        alignn_runs.append(pred.assign(group=gi, member=name))

        mae = metrics(pred)["MAE, K"]
        print(f"group {gi}, {name}: MAE on test {mae:.2f} K, the best epoch {best_epoch}")

alignn_runs = pd.concat(alignn_runs)

alignn = alignn_runs.groupby(["group", "id"], as_index=False).agg(
    Tc_true=("Tc_true", "first"), Tc_pred=("Tc_pred", "mean")
)

## 2. MACE

In [ ]:
def to_ase(atoms):
    if atoms["cartesian"]:
        positions = dict(positions=atoms["coords"])
    else:
        positions = dict(scaled_positions=atoms["coords"])
    return AseAtoms(symbols=atoms["elements"], cell=atoms["lattice_mat"], pbc=True, **positions)

def pooled_mean(vectors, mask):
    if mask.any():
        return vectors[mask].mean(0)
    return np.zeros(vectors.shape[1])

from mace.calculators import mace_mp
calculator = mace_mp(model=cfg["mace"]["model"], device=device.type, default_dtype="float32")
mace = {}
for n, record in enumerate(data):
    elements = record["atoms"]["elements"]
    descriptors = np.asarray(calculator.get_descriptors(to_ase(record["atoms"])), float)
    descriptors = descriptors.reshape(len(elements), -1)
    is_hydrogen = np.array([e == "H" for e in elements])
    all_atoms_mean = descriptors.mean(0)
    hydrogen_mean = pooled_mean(descriptors, is_hydrogen)
    other_mean = pooled_mean(descriptors, ~is_hydrogen)
    mace[record["id"]] = np.concatenate([all_atoms_mean, hydrogen_mean, other_mean])
    if n % 500 == 0:
        print(n, "/", len(data))
del calculator
torch.cuda.empty_cache()
torch.set_default_dtype(torch.float32)

Xm = {}
for i in by_id:
    Xm[i] = np.concatenate([mace[i], extra[i]])

In [ ]:
def train_mace(gi, seed, epochs):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    g = groups[gi]
    xtr = np.array([Xm[i] for i in g["train"]])
    xm = xtr.mean(0)
    xs = xtr.std(0)
    xs = np.where(xs > 1e-6, xs, 1.0)

    def features(ids):
        values = np.array([Xm[i] for i in ids])
        values = np.clip((values - xm) / xs, -10, 10)
        return torch.tensor(values, dtype=torch.float32, device=device)

    tc = tc_values(g["train"])
    mean = float(tc.mean())
    std = float(tc.std() + 1e-8)

    def inverse_transform(values):
        return np.clip(values * std + mean, 0, None)

    x_tr = features(g["train"])
    y_tr = torch.tensor((tc - mean) / std, dtype=torch.float32, device=device)
    x_val = features(g["val"])
    y_val = tc_values(g["val"])

    net = nn.Sequential(
        nn.Dropout(0.1),
        nn.Linear(x_tr.shape[1], 256),
        nn.SiLU(),
        nn.Dropout(0.2),
        nn.Linear(256, 64),
        nn.SiLU(),
        nn.Linear(64, 1),
    ).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-3)

    best_state = None
    best_score = np.inf

    for epoch in range(epochs):
        net.train()
        for batch_ids in torch.randperm(len(x_tr), device=device).split(64):
            loss = nn.functional.huber_loss(net(x_tr[batch_ids]).squeeze(-1), y_tr[batch_ids], delta=cfg["huber_delta"])
            opt.zero_grad()
            loss.backward()
            opt.step()

        net.eval()
        with torch.no_grad():
            prediction = net(x_val).squeeze(-1).cpu().numpy()
            score = float(np.abs(inverse_transform(prediction) - y_val).mean())
        if score < best_score:
            best_state = copy.deepcopy(net.state_dict())
            best_score = score

    net.load_state_dict(best_state)
    net.eval()
    with torch.no_grad():
        prediction = inverse_transform(net(features(g["test"])).squeeze(-1).cpu().numpy())

    return pd.DataFrame({"group": gi, "id": g["test"], "Tc_true": tc_values(g["test"]), "Tc_pred": prediction})

mace_runs = []
for gi in range(len(groups)):
    for seed in cfg["mace"]["seeds"]:
        run_result = train_mace(gi, seed, cfg["mace"]["epochs"])
        run_result = run_result.assign(seed=seed)
        mace_runs.append(run_result)
mace_runs = pd.concat(mace_runs)

mace_pred = mace_runs.groupby(["group", "id"], as_index=False).agg(
    Tc_true=("Tc_true", "first"), Tc_pred=("Tc_pred", "mean")
)

## 3. Results

In [ ]:
w = cfg["mace"]["weight"]
final = alignn.merge(mace_pred[["group", "id", "Tc_pred"]], on=["group", "id"], suffixes=("_alignn", "_mace"))
final["Tc_pred"] = (1 - w) * final.Tc_pred_alignn + w * final.Tc_pred_mace

for column in ["formula", "chemsys", "pressure_GPa"]:
    final[column] = final.id.map(lambda i: by_id[i][column])
final["source"] = final.id.map(lambda i: by_id[i]["provenance"]["source"])
final = final[["id", "formula", "chemsys", "pressure_GPa", "source", "group",
               "Tc_true", "Tc_pred_alignn", "Tc_pred_mace", "Tc_pred"]]
final.to_csv("predictions.csv", index=False)

alignn_only = final.assign(Tc_pred=final.Tc_pred_alignn)
mace_only = final.assign(Tc_pred=final.Tc_pred_mace)
combo_name = f"{1 - w:g} · ALIGNN×3 + {w:g} · MACE"

pd.DataFrame({
    "ALIGNN×3": metrics(alignn_only),
    "MACE": metrics(mace_only),
    combo_name: metrics(final),
}).T.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 5))
ax.scatter(final.Tc_true, final.Tc_pred, s=8, alpha=0.5)
ax.plot([0, 350], [0, 350], "k--", lw=1)
ax.set(xlim=(0, 350), ylim=(0, 350), xlabel="DFT Tc, K", ylabel="predict Tc, K", title="ALIGNN×3 + MACE, test")
plt.show()

In [ ]:
if os.path.exists("/content"):
    files_to_zip = ["predictions.csv"]
    files_to_zip += glob.glob("models/*.pt")
    with zipfile.ZipFile("/content/results.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for name in files_to_zip:
            z.write(name)
    from google.colab import files
    files.download("/content/results.zip")